# Lab 12: Model Selection




What's today?

Even after chossing the hypothesis class, how do we choose the specific model that fits the best?
equivalent and more specific examples:
1. Choose k - number of neighbors in KNN algorithm
2. Choose depth of a tree in decision tree
3. Choose regularization factor 

The key question is, how to evaluate the generalization error (test error). We will go through few schemes:
1. Using the train set
2. Using validation set to evaluate the test error
3. Using cross validation 

In [30]:
import sys 
sys.path.append("../")
from utils import *

In [67]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from scipy.stats import norm
from plotly.subplots import make_subplots

from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [68]:
def oracle_classifier_tree(X):
    labels = []
    for i in range(X.shape[0]):
        labels.append(1 if ((-5 <= X[i][0] <= 0 and -4 <= X[i][1] <= 0) 
                            or (1 <= X[i][0] <= 5 and -1 <= X[i][1] <= 4)
                            or (-3 <= X[i][0] <= 0 and 2 <= X[i][1] <= 5)) else 0)
    return np.array(labels)


X = np.random.uniform(low=-5, high=5, size=(5000, 2))
y = oracle_classifier_tree(X)


go.Figure([go.Scatter(x = X[:, 0], y=X[:, 1], mode = 'markers',
        marker = dict(color=y, colorscale='Portland'))]).show()

In [81]:
def add_noise(X, y, p):
    y_cp = y.copy()
    noisy_ind = np.where(np.random.binomial(1, p, size=X.shape[0]) != 0)
    y_cp[noisy_ind] = (y_cp[noisy_ind] + 1) % 2
    return X, y_cp


X_noisy, y_noisy = add_noise(X[:4000], y[:4000], 0.03)
X_test, y_test = X[4000:], y[4000:]
k_range = range(1, 200, 2) 



go.Figure([go.Scatter(x = X_noisy[:, 0], y=X_noisy[:, 1], mode = 'markers',
        marker = dict(color=y_noisy, colorscale='Portland'))]).show()

# First scheme

Choose the model that minimize the training error:

In [82]:
## training data is all noisy data
X_train, y_train = X_noisy, y_noisy

In [83]:
train_scores = []
test_scores = []
for k in k_range:
    model = KNeighborsClassifier(k)
    model.fit(X_noisy, y_noisy)
    train_scores.append(model.score(X_noisy, y_noisy))
    test_scores.append(model.score(X_test, y_test))
    
    
go.Figure([go.Scatter(x=list(k_range), y=train_scores, mode='markers + lines', marker_color='rgb(152,171,150)', name=r'train score'), 
          go.Scatter(x=list(k_range), y=test_scores, mode='markers + lines', marker_color='rgb(25,115,132)', name=r'test score')])\
.update_layout(title="Scores of K-NN classifier", xaxis_title='k', yaxis_title='accuracy').show()

Note how it looks (by the training error) that the model is best for k=1 (the most complex model).
It's clearly wrong, as k=1 doesn't perform the best, according to the test score.

Let's see if we can do better

# Second scheme

Single Validation set - 
1. Train the models on the training set
2. Evaluate the models on the evaluation test
3. Choose the model that performs the best on the evaluation test

Note:
That estimator of the generalization error is unbiased

In [84]:
X_train, y_train = X_noisy[:3000], y_noisy[:3000]
X_val, y_val = X_noisy[3000:4000], y_noisy[3000:4000]

val_scores = []

for k in k_range:
    model = KNeighborsClassifier(k)
    model.fit(X_train, y_train)
    val_scores.append(model.score(X_val, y_val))
    test_scores.append(model.score(X_test, y_test))

    
go.Figure([go.Scatter(x=list(k_range), y=val_scores, mode='markers + lines', marker_color='rgb(152,171,150)', name=r'val score'), 
          go.Scatter(x=list(k_range), y=test_scores, mode='markers + lines', marker_color='rgb(25,115,132)', name=r'test score')])\
.update_layout(title="Scores of K-NN classifier", xaxis_title='k', yaxis_title='accuracy').show()

# Third Scheme

In [85]:
k_folds = 5

kf = KFold(n_splits=k_folds)

k_scores = []
val_scores = []
test_scores = []

for k in k_range:
    model = KNeighborsClassifier(k)
    model.fit(X_train, y_train)
    val_scores.append(model.score(X_val, y_val))
    test_scores.append(model.score(X_test, y_test))
    model_k_score = []
    for train_index, test_index in kf.split(X_train):
        X_train_folds, y_train_folds = X_train[train_index], y_train[train_index]
        X_eval_folds, y_eval_folds = X_train[test_index], y_train[test_index]
        
        model.fit(X_train_folds, y_train_folds)
        model_k_score.append(model.score(X_eval_folds, y_eval_folds))
        
    k_scores.append(np.mean(np.array(model_k_score)))
    

In [86]:
go.Figure([go.Scatter(x=list(k_range), y=val_scores, mode='markers + lines', marker_color='rgb(152,171,150)', name=r'val score'), 
      go.Scatter(x=list(k_range), y=test_scores, mode='markers + lines', marker_color='rgb(25,115,132)', name=r'test score'),
      go.Scatter(x=list(k_range), y=k_scores, mode='markers + lines', marker_color='rgb(220,179,144)', name=r'CV estimation')])\
.update_layout(title="Scores of K-NN classifier", xaxis_title='k', yaxis_title='accuracy').show()
